In [2]:
# Import libraries

from anthropic import Anthropic
from dotenv import load_dotenv

In [3]:
# Loading Anthropic API Key
load_dotenv()

True

## STEP 1: Creating a Client and asking a question

In [14]:
# Create an API Client
client = Anthropic()

# Defining parameters
model = "claude-haiku-4-5"
max_tokens=1000
prompt="Howdyyyy"

In [15]:
# Send a request
message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": prompt
    }

])

In [6]:
# Retrieve the answer
message.content[0].text

"Howdy! 👋 How's it going? What can I help you with today?"

In [ ]:
# Try multiple messages
message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "Hoowdyyy"
    },
    {
        "role":"user",
        "content": "Can you tell me what time is it?"
    }

])

## A new value under the messages lists provides additional context to the same prompt

In [8]:
# Retrieving the answer
message.content[0].text

'Howdy! 👋\n\nI don\'t have access to real-time information, so I can\'t tell you the current time. However, you can check the time by:\n\n- Looking at your device (phone, computer, watch)\n- Asking a voice assistant like Siri, Alexa, or Google Assistant\n- Searching "current time" online\n\nIs there anything else I can help you with?'

In [ ]:
# Trying user-assistant multi-conversation
# The LLM doesn't store the messages by default. 
# Here, I'm simulating how Claude would behave if we store interactions under the same message list.

message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "What is an apple?"
    },
{
        "role":"assistant",
        "content": "An apple is a red fruit"
    },
{
        "role":"user",
        "content": "What about green ones?"
    }

])


In [10]:
# Retrieving the answer
message.content[0].text

"You're right to point that out! Apples come in various colors, including:\n\n- **Red** apples (like Red Delicious, Gala)\n- **Green** apples (like Granny Smith)\n- **Yellow** apples (like Golden Delicious)\n- **Mixed colors** (like Fuji or Honeycrisp)\n\nSo my initial answer was incomplete. Apples are fruits that grow on apple trees and come in multiple colors, each variety having different tastes and uses—some are sweeter, some more tart."

## STEP 2: Storing message interactions

In [ ]:
# Creating functions to maintain context for conversations

# messages it's a list that will be defined when calling the function
# add_user_message adds the user prompt do the message list whenever the user asks something
# add_assistant message does the same, but whenever the LLM answers something
# the chat() function only puts the request inside a function

def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=max_tokens,
        messages=messages
    )
    return message.content[0].text

In [16]:
# Creating list - it will append the conversation history

messages = []

def conversation(messages):
    while True:
        try: 
            user_prompt = input("Prompt: ") ## input() is a function to prompt the user to insert a message
            if user_prompt.lower() == 'exit': # handling errors when prompting
                break
        except KeyboardInterrupt:
            print('Assistant: Goodbye')
            break
        except EOFError:
            print('Assistant: Goodbye')
            break
                                            ## executing the interaction flow
        add_user_message(messages, user_prompt)
        print(f"User: {user_prompt}")
        answer = chat(messages)
        print(f"Assistant: {answer}")
        add_assistant_message(messages, answer)



In [18]:
conversation(messages)

User: Could you list 5 types of apples to me?
Assistant: # 5 Types of Apples

1. **Gala** — Sweet and mild, with a thin skin. Great for eating fresh.

2. **Granny Smith** — Bright green, tart, and crisp. Popular for baking and cooking.

3. **Fuji** — Sweet, dense, and crispy. One of the most popular eating apples.

4. **Honeycrisp** — Very crisp and juicy with a balanced sweet-tart flavor. Excellent for fresh eating.

5. **Red Delicious** — Deep red, sweet with mild tartness. A classic eating apple.

Is there anything specific you'd like to know about these apples?
User: What about coconuts?
Assistant: # 5 Types of Coconuts

1. **Green Coconut** — Young coconuts with soft, jelly-like flesh inside. High in coconut water. Common in tropical regions for fresh drinking.

2. **Brown Coconut** — Mature coconuts with harder shell and denser white meat. Used for coconut milk, oil, and dried coconut products.

3. **Dwarf Coconut** — Smaller variety that matures quickly. Commonly grown in home g

In [ ]:
# For agents, a good practice is to initialize the conversation with a system message 
#to give more context to the LLM about their role. Example: '''

messages = [
    {
        "role": "user", 
        "content": "You are a helpful assistant specialized in botanic."
    }
]
conversation(messages)

User: Do you have any tip for me?
Assistant: # Botany Tips for You! 🌱

I'd be happy to help! Here are some universally useful tips:

## **General Gardening/Plant Care**
- **Know your light** - Assess how much sun/shade your space gets before choosing plants
- **Check soil drainage** - Most plants prefer well-draining soil; use a simple percolation test
- **Water wisely** - Overwatering kills more plants than underwatering. Check soil moisture before watering
- **Observe seasonally** - Plants have different needs in different seasons

## **For Plant Identification**
- Learn your plant's **family, genus, and species** - this reveals a lot about its needs
- Notice **leaf shape, arrangement, and texture** - key identification features

## **For Growing Success**
- Start with hardy, low-maintenance plants (pothos, snake plants, succulents)
- Group plants with similar water needs together
- Repot when roots circle the soil surface

---

**But I'd love to help more specifically!** Could you t

In [ ]:
''' Or the request can take a system parameter, where you input a system message.'''

# Send a request with a system prompt
system_prompt="You are a helpful assistant specialized in botanic."

message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "Do you have any tip for me?"
    },],
    system=system_prompt
    )
message.content[0].text

"# Tips for Botany Interests\n\nI'd be happy to help! Here are some general botany tips:\n\n## For Plant Care\n- **Water wisely** - Most plants prefer soil that's moist but not waterlogged\n- **Light matters** - Know your plant's light requirements before placing it\n- **Humidity helps** - Many tropical plants appreciate misting or a pebble tray\n- **Repot when needed** - Usually when roots grow through drainage holes\n\n## For Learning Botany\n- **Start with common plants** - They're easier to observe and maintain\n- **Keep a plant journal** - Track growth, changes, and care routines\n- **Join a community** - Plant societies and online groups are great resources\n- **Visit botanical gardens** - See diverse species and get care ideas\n\n## General Wisdom\n- **Patience pays off** - Plants grow on their own timeline\n- **Failure teaches** - Dead plants teach you what not to do next time\n- **Read labels carefully** - Plant tags contain crucial care information\n\n---\n\n**Is there a spec

In [ ]:
'''Testing different temperatures.
0: Deterministic output - tasks that are factual
1: Random output - tasks that are creative
'''

system_prompt="You are a helpful assistant specialized in botanic."

message = client.messages.create(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "Do you have any tip for me?"
    },],
    system=system_prompt,
    temperature=1
    )
message.content[0].text

"# Happy to help! Here are some general tips:\n\n**General wellness tips:**\n- Stay hydrated throughout the day\n- Get regular sunlight exposure\n- Move your body regularly\n- Get adequate sleep\n\n**If you're interested in botany specifically:**\n- Start a small indoor garden or herb collection—it's rewarding and low-pressure\n- Learn to identify common plants in your area on walks\n- Keep a plant journal or photo collection\n- Research plants that thrive in your climate zone\n- Join local gardening or naturalist groups\n\n**For plant care at home:**\n- Know your light conditions before buying plants\n- Don't overwater—most houseplant deaths come from this\n- Use well-draining soil\n- Start with hardy, forgiving plants like pothos, snake plants, or ZZ plants\n\nIs there something specific you'd like tips about? I'm especially equipped to help with plant-related questions! 🌱"

In [ ]:
'''Implementing Response Streaming
    It outputs the text while it is generate, instead all at once.
'''

stream = client.messages.stream(
    model=model,
    max_tokens=max_tokens,
    messages=[{
        "role":"user",
        "content": "Do you have any tip for me?"
    },],
    system=system_prompt,
    temperature=1,
    )

with stream as stream:
    for text in stream.text_stream:
        print(text, end="")

# General Botany Tips

I'd be happy to help! Here are some universally useful tips:

## **For Plant Care:**
- **Water wisely** – Most plants prefer drying out slightly between waterings rather than staying constantly wet
- **Light matters** – Know your plant's light requirements; low light is a common killer
- **Good drainage** – Use well-draining soil and pots with drainage holes
- **Humidity** – Many tropical plants appreciate misting or a pebble tray with water

## **For Plant Identification:**
- Look at leaf shape, arrangement, and texture
- Note flower characteristics (color, petal count, arrangement)
- Check for distinctive features like thorns, fragrance, or seed pods
- Use plant ID apps if you're stuck

## **For Growing Plants:**
- Start with hardy, forgiving species (pothos, snake plants, ZZ plants)
- Match plants to your home's conditions rather than fighting them
- Observe seasonal changes – many plants have dormancy periods

---

**Is there something more specific you'd lik

In [ ]:
response = stream.get_final_message()
response

ParsedMessage(id='msg_01VHNwWyjhXLWGQo9kLeMZ1q', container=None, content=[ParsedTextBlock(citations=None, text="# General Botany Tips\n\nI'd be happy to help! Here are some universally useful tips:\n\n## **For Plant Care:**\n- **Water wisely** – Most plants prefer drying out slightly between waterings rather than staying constantly wet\n- **Light matters** – Know your plant's light requirements; low light is a common killer\n- **Good drainage** – Use well-draining soil and pots with drainage holes\n- **Humidity** – Many tropical plants appreciate misting or a pebble tray with water\n\n## **For Plant Identification:**\n- Look at leaf shape, arrangement, and texture\n- Note flower characteristics (color, petal count, arrangement)\n- Check for distinctive features like thorns, fragrance, or seed pods\n- Use plant ID apps if you're stuck\n\n## **For Growing Plants:**\n- Start with hardy, forgiving species (pothos, snake plants, ZZ plants)\n- Match plants to your home's conditions rather th